# 02 · Is this customer ready? (REAL dataset)
Readiness scorecard produced by the `adequacy` toolkit (`src/adequacy/readiness.py`, metrics in `sql/02_adequacy.sql`).

In [1]:
from pathlib import Path
import json
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
pd.set_option('display.width', 160); pd.set_option('display.max_colwidth', 80)

In [2]:
m = json.loads((ROOT / 'data/public/intake/metrics.json').read_text())
print(m['verdict'], m['score'])
{k: m[k] for k in ['days_covered', 'median_events_per_day', 'share_users_2plus', 'share_sessions_2plus', 'density', 'share_catalog_under5', 'share_top1pct', 'new_items_per_day']}

GO WITH CONDITIONS 73


{'days_covered': 139.0,
 'median_events_per_day': 19836.0,
 'share_users_2plus': 0.28763539605455274,
 'share_sessions_2plus': 0.21473251876948898,
 'density': 6.383453433015056e-06,
 'share_catalog_under5': 0.8129110688569559,
 'share_top1pct': 0.22818923266511765,
 'new_items_per_day': 1271.233870967742}

In [3]:
pd.read_csv(ROOT / 'data/public/intake/readiness.csv')[['grade', 'criterion', 'value', 'threshold', 'critical']]

,grade,criterion,value,threshold,critical
0,PASS,History length (days),139,pass >= 56; fail < 28,True
1,CONDITION,Median events per day,"19,836","pass >= 50,000; fail < 5,000",False
2,CONDITION,Users with 2+ events,29%,pass >= 40%; fail < 20%,False
3,CONDITION,Sessions with 2+ events,21%,pass >= 40%; fail < 20%,True
4,CONDITION,Events on items missing from catalog,9.3%,pass <= 1.0%; fail >= 10.0%,True
5,PASS,Catalog items with a category,100%,pass >= 95%; fail < 70%,False
6,PASS,"Events beyond views (cart, purchase)",2.9%,pass >= 1.0%; fail < 0.2%,False
7,CONDITION,Catalog items with < 5 interactions,81%,pass <= 50%; above: condition,False
8,PASS,Blocking intake checks,0,"pass = 0, else fail",True


## The message the toolkit drafts for the customer

In [4]:
print((ROOT / 'data/public/intake/customer_message.txt').read_text())

Hi both,

Thanks for the data drop. We validated 2,657,575 events from 1,407,445 users over 139 days.
Our readiness verdict is GO WITH CONDITIONS (score 73/100).
We can start training now. In parallel, could you help us with the points below before the A/B test starts (week 5)?
- Median events per day (19,836): Add more surfaces or event types (clicks on all pages, not only the homepage).
- Users with 2+ events (29%): Send a stable user id (logged-in id or long-lived cookie) so visits link together.
- Sessions with 2+ events (21%): Check that every page view / click is tracked, not only the first one.
- Events on items missing from catalog (9.3%): Send a full catalog export, including sold and delisted items.
- Catalog items with < 5 interactions (81%): Make sure every listing has a title, category and image URL: content is what makes the long tail and brand-new items recommendable.

The full intake report is attached. Shall we review it in Thursday's check-in?

Best,


## Run the toolkit yourself on the synthetic example
`python -m adequacy check --events examples/events.csv --catalog examples/catalog.csv --out reports`